# Database Validation
SQL queries against `war_games.db` to verify correctness.

In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect('../databases/war_games.db')
conn.execute('PRAGMA foreign_keys = ON')

def q(sql):
    return pd.read_sql_query(sql, conn)

## 1. Row counts

In [2]:
for table in ['People', 'Teams', 'Salaries', 'Batting', 'Pitching']:
    count = q(f'SELECT COUNT(*) AS n FROM {table}').iloc[0, 0]
    print(f'{table}: {count:,} rows')

People: 24,270 rows
Teams: 3,614 rows
Salaries: 26,428 rows
Batting_with_WAR: 25,132 rows
Pitching_with_WAR: 20,455 rows


## 2. WAR coverage (should be 100%)

In [3]:
q("""
SELECT
    'Batting' AS table_name,
    COUNT(*) AS total,
    SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) AS with_war,
    ROUND(100.0 * SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct
FROM Batting
UNION ALL
SELECT
    'Pitching',
    COUNT(*),
    SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END),
    ROUND(100.0 * SUM(CASE WHEN WAR IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 1)
FROM Pitching
""")

,table_name,total,with_war,pct
0,Batting,25132,25132,100.0
1,Pitching,20455,20455,100.0


## 3. Spot-check: Aaron Judge

In [4]:
q("""
SELECT b.yearID, b.teamID, t.name AS team_name,
       b.G, b.AB, b.HR, b.RBI, b.WAR
FROM Batting b
JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
WHERE b.playerID = 'judgeaa01'
ORDER BY b.yearID
""")

,yearID,teamID,team_name,G,AB,HR,RBI,WAR
0,2016,NYA,New York Yankees,27,84,4,10,-0.3
1,2017,NYA,New York Yankees,155,542,52,114,8.1
2,2018,NYA,New York Yankees,112,413,27,67,6.0
3,2019,NYA,New York Yankees,102,378,27,55,5.6
4,2020,NYA,New York Yankees,28,101,9,22,1.1
5,2021,NYA,New York Yankees,148,550,39,98,5.9
6,2022,NYA,New York Yankees,157,570,62,131,10.8
7,2023,NYA,New York Yankees,106,367,37,75,4.6
8,2024,NYA,New York Yankees,158,559,58,144,10.9
9,2025,NYA,New York Yankees,152,541,53,114,9.7


## 4. Spot-check: Ohtani (batting + pitching)

In [5]:
print('Batting:')
display(q("""
SELECT b.yearID, b.teamID, b.G, b.AB, b.HR, b.WAR
FROM Batting b
WHERE b.playerID = 'ohtansh01'
ORDER BY b.yearID
"""))

print('Pitching:')
display(q("""
SELECT p.yearID, p.teamID, p.G, p.W, p.L, p.ERA, p.WAR
FROM Pitching p
WHERE p.playerID = 'ohtansh01'
ORDER BY p.yearID
"""))

Batting:


,yearID,teamID,G,AB,HR,WAR
0,2018,LAA,114,326,22,2.7
1,2019,LAA,106,384,18,2.4
2,2020,LAA,46,153,7,0.0
3,2021,LAA,158,537,46,4.9
4,2022,LAA,157,586,34,3.4
5,2023,LAA,135,497,44,6.1
6,2024,LAN,159,636,54,9.0
7,2025,LAN,158,611,55,6.6


Pitching:


,yearID,teamID,G,W,L,ERA,WAR
0,2018,LAA,10,4,2,3.31,1.3
1,2020,LAA,2,0,1,37.80,-0.4
2,2021,LAA,23,9,2,3.18,4.1
3,2022,LAA,28,15,9,2.33,6.3
4,2023,LAA,23,10,5,3.14,3.8
5,2025,LAN,14,1,1,2.87,1.1


## 5. Multi-team player
Verify a traded player has separate rows per team with WAR.

In [6]:
q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.yearID, b.stint, b.teamID, b.G, b.AB, b.HR, b.WAR
FROM Batting b
JOIN People p ON b.playerID = p.playerID
WHERE b.playerID IN (
    SELECT playerID FROM Batting
    WHERE yearID = 2024
    GROUP BY playerID, yearID
    HAVING COUNT(*) > 1
    LIMIT 1
)
AND b.yearID = 2024
ORDER BY b.stint
""")

,name,yearID,stint,teamID,G,AB,HR,WAR
0,Nick Ahmed,2024,1,SFN,52,155,1,-0.2
1,Nick Ahmed,2024,2,LAN,17,48,1,-0.1
2,Nick Ahmed,2024,3,SDN,2,7,0,-0.2


## 6. Join integrity: Batting → People → Teams

In [7]:
# Batting rows with no matching People record
orphan_people = q("""
SELECT COUNT(*) AS orphans FROM Batting b
LEFT JOIN People p ON b.playerID = p.playerID
WHERE p.playerID IS NULL
""")
print(f'Batting rows with no People match: {orphan_people.iloc[0,0]}')

# Batting rows with no matching Teams record
orphan_teams = q("""
SELECT COUNT(*) AS orphans FROM Batting b
LEFT JOIN Teams t ON b.teamID = t.teamID AND b.yearID = t.yearID
WHERE t.teamID IS NULL
""")
print(f'Batting rows with no Teams match: {orphan_teams.iloc[0,0]}')

Batting rows with no People match: 0
Batting rows with no Teams match: 0


## 7. Top 5 batting WAR per year

In [8]:
q("""
SELECT yearID, name, teamID, HR, WAR FROM (
    SELECT b.yearID, p.nameFirst || ' ' || p.nameLast AS name,
           b.teamID, b.HR, b.WAR,
           RANK() OVER (PARTITION BY b.yearID ORDER BY b.WAR DESC) AS rnk
    FROM Batting b
    JOIN People p ON b.playerID = p.playerID
    WHERE b.yearID IN (2020, 2022, 2024)
)
WHERE rnk <= 5
ORDER BY yearID, WAR DESC
""")

,yearID,name,teamID,HR,WAR
0,2020,Mookie Betts,LAN,16,3.7
1,2020,Freddie Freeman,ATL,13,3.3
2,2020,DJ LeMahieu,NYA,10,3.0
3,2020,Manny Machado,SDN,16,3.0
4,2020,Marcell Ozuna,ATL,18,2.8
5,2020,Dansby Swanson,ATL,10,2.8
6,2020,Fernando Tatis,SDN,17,2.8
7,2020,Trea Turner,WAS,12,2.8
8,2022,Aaron Judge,NYA,62,10.8
9,2022,Nolan Arenado,SLN,30,7.9


## 8. Top 5 pitching WAR per year

In [9]:
q("""
SELECT yearID, name, teamID, W, ERA, WAR FROM (
    SELECT p2.yearID, pe.nameFirst || ' ' || pe.nameLast AS name,
           p2.teamID, p2.W, p2.ERA, p2.WAR,
           RANK() OVER (PARTITION BY p2.yearID ORDER BY p2.WAR DESC) AS rnk
    FROM Pitching p2
    JOIN People pe ON p2.playerID = pe.playerID
    WHERE p2.yearID IN (2020, 2022, 2024)
)
WHERE rnk <= 5
ORDER BY yearID, WAR DESC
""")

,yearID,name,teamID,W,ERA,WAR
0,2020,Shane Bieber,CLE,8,1.63,3.2
1,2020,Trevor Bauer,CIN,5,1.73,3.0
2,2020,Max Fried,ATL,7,2.25,2.9
3,2020,Hyun Jin Ryu,TOR,5,2.69,2.9
4,2020,Yu Darvish,CHN,8,2.01,2.8
5,2020,Zack Wheeler,PHI,4,2.92,2.8
6,2022,Sandy Alcantara,MIA,14,2.28,8.0
7,2022,Dylan Cease,CHA,14,2.20,6.4
8,2022,Shohei Ohtani,LAA,15,2.33,6.3
9,2022,Max Fried,ATL,14,2.48,6.0


## 9. Highest-paid vs highest WAR (2015)

In [10]:
print('Top 10 paid batters (2015):')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       s.teamID, s.salary, b.WAR
FROM Salaries s
JOIN People p ON s.playerID = p.playerID
JOIN Batting b ON s.playerID = b.playerID
    AND s.yearID = b.yearID AND s.teamID = b.teamID
WHERE s.yearID = 2015
ORDER BY s.salary DESC
LIMIT 10
"""))

print('Top 10 WAR batters (2015):')
display(q("""
SELECT p.nameFirst || ' ' || p.nameLast AS name,
       b.teamID, b.WAR, s.salary
FROM Batting b
JOIN People p ON b.playerID = p.playerID
LEFT JOIN Salaries s ON b.playerID = s.playerID
    AND b.yearID = s.yearID AND b.teamID = s.teamID
WHERE b.yearID = 2015
ORDER BY b.WAR DESC
LIMIT 10
"""))

Top 10 paid batters (2015):


,name,teamID,salary,WAR
0,Clayton Kershaw,LAN,32571000.0,0.0
1,Zack Greinke,LAN,25000000.0,0.6
2,Ryan Howard,PHI,25000000.0,-1.7
3,Felix Hernandez,SEA,24857000.0,0.0
4,Albert Pujols,LAA,24000000.0,3.1
5,Robinson Cano,SEA,24000000.0,3.8
6,Prince Fielder,TEX,24000000.0,1.8
7,Cole Hamels,PHI,23500000.0,-0.1
8,Mark Teixeira,NYA,23125000.0,3.2
9,Joe Mauer,MIN,23000000.0,1.5


Top 10 WAR batters (2015):


,name,teamID,WAR,salary
0,Bryce Harper,WAS,9.7,2500000.0
1,Mike Trout,LAA,9.5,6083000.0
2,Paul Goldschmidt,ARI,8.3,3100000.0
3,Joey Votto,CIN,7.7,14000000.0
4,Josh Donaldson,TOR,7.4,4300000.0
5,Manny Machado,BAL,7.3,548000.0
6,Jason Heyward,SLN,7.0,7800000.0
7,Lorenzo Cain,KCA,6.9,2725000.0
8,Kevin Kiermaier,TBA,6.9,513800.0
9,AJ Pollock,ARI,6.8,519500.0
